# Data Analyst Jobs — Exploration

Quick EDA of the enriched dataset before loading into Tableau.

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path("..")
ENRICHED = ROOT / "data" / "enriched" / "jobs_enriched.csv"
DB = ROOT / "jobs.db"

## Load enriched CSV

In [ ]:
df = pd.read_csv(ENRICHED)
print(f"Rows: {len(df)}")
df.head(3)

## Column overview

In [ ]:
df.info()

## Seniority distribution

In [ ]:
df["seniority"].value_counts()

## Remote policy distribution

In [ ]:
df["remote_policy"].value_counts()

## Top required skills

In [ ]:
skills = df["required_skills"].dropna().apply(json.loads).explode()
skills.str.strip().str.lower().value_counts().head(20)

## Salary distribution (where available)

In [ ]:
salary = df[["min_amount", "max_amount"]].dropna(how="all")
salary.describe()

## Query the database

In [ ]:
if DB.exists():
    con = sqlite3.connect(DB)
    top_skills = pd.read_sql(
        "SELECT skill, COUNT(*) AS n FROM skills WHERE skill_type='required' "
        "GROUP BY skill ORDER BY n DESC LIMIT 20",
        con,
    )
    con.close()
    display(top_skills)
else:
    print("Run db.py first to generate jobs.db")